<a href="https://colab.research.google.com/github/springboardmentor123g/PlantDocBot/blob/intern-AnshikaSahu/Text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
import numpy as np
import evaluate


In [3]:
df = pd.read_parquet("hf://datasets/ButterChicken98/plantvillage-image-text-pairs/data/train-00000-of-00001.parquet")
df = df.drop(columns=["image"])
df = df.explode("captions")
df = df.rename(columns={"captions": "text"})

label_col = "caption"
text_col = "text"

encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df[label_col].tolist())
num_labels = len(encoder.classes_)

df_train, df_test = train_test_split(
    df, train_size=0.8, random_state=42, stratify=df["label"]
)
train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(data):
    return tokenizer(data[text_col], truncation=True)

tokenized_train = train_dataset.map(tokenize_fn, batched=True)
tokenized_test = test_dataset.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/66041 [00:00<?, ? examples/s]

Map:   0%|          | 0/16511 [00:00<?, ? examples/s]

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return metric.compute(predictions=predictions, references=labels)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-2225597763.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [7]:
trainer.train()

results = trainer.evaluate()
print("Final evaluation:", results)

# Save the best model
trainer.save_model("./best_plant_text_classifier")
tokenizer.save_pretrained("./best_plant_text_classifier")

Epoch,Training Loss,Validation Loss,Accuracy
1,0.000100,0.000047,1.000000
2,0.000000,0.000006,1.000000
3,0.000000,0.000002,1.000000


Final evaluation: {'eval_loss': 4.740727672469802e-05, 'eval_accuracy': 1.0, 'eval_runtime': 14.8165, 'eval_samples_per_second': 1114.364, 'eval_steps_per_second': 69.652, 'epoch': 3.0}


('./best_plant_text_classifier/tokenizer_config.json',
 './best_plant_text_classifier/special_tokens_map.json',
 './best_plant_text_classifier/vocab.txt',
 './best_plant_text_classifier/added_tokens.json',
 './best_plant_text_classifier/tokenizer.json')

In [8]:
loaded_model = AutoModelForSequenceClassification.from_pretrained("./best_plant_text_classifier")
loaded_tokenizer = AutoTokenizer.from_pretrained("./best_plant_text_classifier")

sample_text = "leaf has brown spots and looks dry"
inputs = loaded_tokenizer(sample_text, return_tensors="pt", truncation=True, padding=True)

with torch.no_grad():
    outputs = loaded_model(**inputs)
    prediction = outputs.logits.argmax(-1).item()

predicted_label = encoder.inverse_transform([prediction])[0]
print(f"Sample text: {sample_text}")
print(f"Predicted label: {predicted_label}")


Sample text: leaf has brown spots and looks dry
Predicted label: Tomato Septoria leaf spot


In [14]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [15]:
model.save_pretrained("/content/drive/MyDrive/best_plant_text_classifier")
tokenizer.save_pretrained("/content/drive/MyDrive/best_plant_text_classifier")


('/content/drive/MyDrive/best_plant_text_classifier/tokenizer_config.json',
 '/content/drive/MyDrive/best_plant_text_classifier/special_tokens_map.json',
 '/content/drive/MyDrive/best_plant_text_classifier/vocab.txt',
 '/content/drive/MyDrive/best_plant_text_classifier/added_tokens.json',
 '/content/drive/MyDrive/best_plant_text_classifier/tokenizer.json')

In [16]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("/content/drive/MyDrive/best_plant_text_classifier")
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/best_plant_text_classifier")


In [18]:
import os
os.listdir("/content/drive/MyDrive/best_plant_text_classifier")


['config.json',
 'model.safetensors',
 'tokenizer_config.json',
 'special_tokens_map.json',
 'vocab.txt',
 'tokenizer.json']